In [6]:
import notebookutils
from pyspark.sql.functions import (
    col, when, lit, current_timestamp,
    count, sum as spark_sum, avg, round as spark_round,
    countDistinct, max as spark_max, min as spark_min,
    datediff, to_date
)
from datetime import datetime

SILVER_PATH   = "abfss://bddf1263-676e-44e7-bf20-6d7b218dc2d2@onelake.dfs.fabric.microsoft.com/c49fa9c5-5ba8-4add-a7df-fa808a6fa426"
GOLD_PATH     = "abfss://bddf1263-676e-44e7-bf20-6d7b218dc2d2@onelake.dfs.fabric.microsoft.com/07bd3e33-6f0c-4dac-b2a4-b70dca6c6298"
SILVER_TABLES = f"{SILVER_PATH}/Tables/dbo"
GOLD_TABLES   = f"{GOLD_PATH}/Tables/dbo"
PIPELINE_NAME = "NB_10_Gold_Analytics"
BATCH_DATE    = datetime.now().strftime("%Y-%m-%d")

print(f"Gold Analytics Pipeline")
print(f"Started: {datetime.now()}")

StatementMeta(, 876dd5de-bf96-4a12-8cce-079ce232d822, 8, Finished, Available, Finished, False)

Gold Analytics Pipeline
Started: 2026-05-06 01:19:37.499855


In [7]:
# ── Read Silver tables ───────────────────────────────────────
df_transactions  = spark.read.format("delta").load(f"{SILVER_TABLES}/silver_transactions")
df_merchants     = spark.read.format("delta").load(f"{SILVER_TABLES}/silver_merchants")
df_cardholders   = spark.read.format("delta").load(f"{SILVER_TABLES}/silver_cardholders")
df_settlements   = spark.read.format("delta").load(f"{SILVER_TABLES}/silver_settlements")
df_disputes      = spark.read.format("delta").load(f"{SILVER_TABLES}/silver_disputes")
df_fraud         = spark.read.format("delta").load(f"{SILVER_TABLES}/silver_fraud_labels")

print(f"Transactions : {df_transactions.count():,}")
print(f"Merchants    : {df_merchants.count():,}")
print(f"Cardholders  : {df_cardholders.count():,}")
print(f"Settlements  : {df_settlements.count():,}")
print(f"Disputes     : {df_disputes.count():,}")
print(f"Fraud labels : {df_fraud.count():,}")

StatementMeta(, 876dd5de-bf96-4a12-8cce-079ce232d822, 9, Finished, Available, Finished, False)

Transactions : 145,946
Merchants    : 2,000
Cardholders  : 5,000
Settlements  : 34,561
Disputes     : 3,500
Fraud labels : 2,800


In [8]:
# ── Gold Merchant Performance ────────────────────────────────
df_merchant_perf = (df_transactions
    .join(df_merchants.select(
        "merchant_id", "merchant_name", "category_code",
        "category_name", "province", "city",
        "risk_rating", "status", "acquiring_bank"),
        "merchant_id", "left")
    .join(df_settlements.select(
        "merchant_id",
        col("gross_amount_cad").alias("settlement_gross"),
        col("net_settlement_amount_cad").alias("settlement_net"),
        col("chargeback_debit_cad").alias("chargeback_amount"),
        col("status").alias("settlement_status")),
        "merchant_id", "left")
    .groupBy(
        "merchant_id", "merchant_name", "category_code",
        "category_name", "province", "city",
        "risk_rating", "status", "acquiring_bank")
    .agg(
        count("transaction_id").alias("total_transactions"),
        spark_sum("amount_cad").alias("total_volume_cad"),
        avg("amount_cad").alias("avg_transaction_cad"),
        spark_sum(when(col("is_approved") == "Y", 1).otherwise(0))
            .alias("approved_count"),
        spark_sum(when(col("is_approved") == "N", 1).otherwise(0))
            .alias("declined_count"),
        spark_sum(when(col("is_flagged") == "Y", 1).otherwise(0))
            .alias("flagged_count"),
        spark_sum(when(col("is_fraud_case") == "Y", 1).otherwise(0))
            .alias("fraud_count") if "is_fraud_case" in df_transactions.columns
            else lit(0).alias("fraud_count"),
        spark_sum("interchange_fee_cad").alias("total_interchange_fees_cad"),
        spark_sum("settlement_gross").alias("total_settlement_gross_cad"),
        spark_sum("chargeback_amount").alias("total_chargeback_cad")
    )
    .withColumn("approval_rate_pct",
        spark_round(col("approved_count") / col("total_transactions") * 100, 2))
    .withColumn("avg_transaction_cad",
        spark_round(col("avg_transaction_cad"), 2))
    .withColumn("total_volume_cad",
        spark_round(col("total_volume_cad"), 2))
    .withColumn("total_interchange_fees_cad",
        spark_round(col("total_interchange_fees_cad"), 2))
    .withColumn("_gold_loaded_at", current_timestamp())
    .withColumn("_pipeline_name",  lit(PIPELINE_NAME))
    .withColumn("_batch_date",     lit(BATCH_DATE))
)

(df_merchant_perf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{GOLD_TABLES}/gold_merchant_performance"))

count_mp = spark.read.format("delta").load(
    f"{GOLD_TABLES}/gold_merchant_performance").count()
print(f"gold_merchant_performance: {count_mp:,} rows ✅")

StatementMeta(, 876dd5de-bf96-4a12-8cce-079ce232d822, 10, Finished, Available, Finished, False)

gold_merchant_performance: 2,000 rows ✅


In [9]:
# ── Gold Fraud Summary ───────────────────────────────────────
df_fraud_summary = (df_fraud
    .join(df_cardholders.select(
        "cardholder_id", "province",
        "issuing_bank", "risk_rating",
        "income_band", "credit_band"),
        "cardholder_id", "left")
    .groupBy(
        "fraud_type", "detection_method",
        "confirmation_status", "severity",
        "province", "issuing_bank")
    .agg(
        count("fraud_case_id").alias("total_cases"),
        spark_sum("amount_at_risk_cad").alias("total_amount_at_risk_cad"),
        spark_sum("amount_lost_cad").alias("total_amount_lost_cad"),
        spark_sum("recovered_amount_cad").alias("total_recovered_cad"),
        avg("ml_fraud_score").alias("avg_ml_fraud_score"),
        spark_sum(when(col("is_confirmed_fraud") == "Y", 1).otherwise(0))
            .alias("confirmed_fraud_count"),
        spark_sum(when(col("is_false_positive") == "Y", 1).otherwise(0))
            .alias("false_positive_count"),
        spark_sum(when(col("is_recovered") == "Y", 1).otherwise(0))
            .alias("recovered_count"),
        spark_sum(when(col("case_closed") == "Y", 1).otherwise(0))
            .alias("closed_count")
    )
    .withColumn("total_amount_at_risk_cad",
        spark_round(col("total_amount_at_risk_cad"), 2))
    .withColumn("total_amount_lost_cad",
        spark_round(col("total_amount_lost_cad"), 2))
    .withColumn("total_recovered_cad",
        spark_round(col("total_recovered_cad"), 2))
    .withColumn("net_loss_cad",
        spark_round(col("total_amount_lost_cad") -
                    col("total_recovered_cad"), 2))
    .withColumn("recovery_rate_pct",
        spark_round(col("total_recovered_cad") /
                    col("total_amount_lost_cad") * 100, 2))
    .withColumn("avg_ml_fraud_score",
        spark_round(col("avg_ml_fraud_score"), 4))
    .withColumn("_gold_loaded_at", current_timestamp())
    .withColumn("_pipeline_name",  lit(PIPELINE_NAME))
    .withColumn("_batch_date",     lit(BATCH_DATE))
)

(df_fraud_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{GOLD_TABLES}/gold_fraud_summary"))

count_fs = spark.read.format("delta").load(
    f"{GOLD_TABLES}/gold_fraud_summary").count()
print(f"gold_fraud_summary: {count_fs:,} rows ✅")

StatementMeta(, 876dd5de-bf96-4a12-8cce-079ce232d822, 11, Finished, Available, Finished, False)

gold_fraud_summary: 2,680 rows ✅


In [10]:
# ── Gold Settlement Summary ──────────────────────────────────
merchants_slim = df_merchants.select(
    "merchant_id",
    col("merchant_name").alias("mer_name"),
    col("category_code").alias("mer_category_code"),
    col("category_name").alias("mer_category_name"),
    col("province").alias("mer_province"),
    col("acquiring_bank").alias("mer_acquiring_bank")
)

df_settlement_summary = (df_settlements
    .join(merchants_slim, "merchant_id", "left")
    .groupBy(
        "settlement_month",
        col("acquiring_bank").alias("acquiring_bank"),
        col("mer_category_code").alias("category_code"),
        col("mer_category_name").alias("category_name"),
        col("mer_province").alias("province"))
    .agg(
        count("settlement_id").alias("total_settlements"),
        spark_sum("transaction_count").alias("total_transactions"),
        spark_sum("gross_amount_cad").alias("total_gross_cad"),
        spark_sum("net_settlement_amount_cad").alias("total_net_cad"),
        spark_sum("interchange_fees_cad").alias("total_interchange_cad"),
        spark_sum("network_fees_cad").alias("total_network_fees_cad"),
        spark_sum("chargeback_debit_cad").alias("total_chargebacks_cad"),
        spark_sum(when(col("status") == "COMPLETED", 1).otherwise(0))
            .alias("completed_count"),
        spark_sum(when(col("status") == "FAILED", 1).otherwise(0))
            .alias("failed_count"),
        spark_sum(when(col("status") == "DISPUTED", 1).otherwise(0))
            .alias("disputed_count"),
        avg("fee_rate_pct").alias("avg_fee_rate_pct")
    )
    .withColumn("total_gross_cad",
        spark_round(col("total_gross_cad"), 2))
    .withColumn("total_net_cad",
        spark_round(col("total_net_cad"), 2))
    .withColumn("total_interchange_cad",
        spark_round(col("total_interchange_cad"), 2))
    .withColumn("total_chargebacks_cad",
        spark_round(col("total_chargebacks_cad"), 2))
    .withColumn("avg_fee_rate_pct",
        spark_round(col("avg_fee_rate_pct"), 4))
    .withColumn("completion_rate_pct",
        spark_round(col("completed_count") /
                    col("total_settlements") * 100, 2))
    .withColumn("_gold_loaded_at", current_timestamp())
    .withColumn("_pipeline_name",  lit(PIPELINE_NAME))
    .withColumn("_batch_date",     lit(BATCH_DATE))
)

(df_settlement_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{GOLD_TABLES}/gold_settlement_summary"))

count_ss = spark.read.format("delta").load(
    f"{GOLD_TABLES}/gold_settlement_summary").count()
print(f"gold_settlement_summary: {count_ss:,} rows ✅")

StatementMeta(, 876dd5de-bf96-4a12-8cce-079ce232d822, 12, Finished, Available, Finished, False)

gold_settlement_summary: 9,207 rows ✅


In [11]:
# ── Final Gold Layer Summary ─────────────────────────────────
print("\n" + "="*60)
print("GOLD LAYER COMPLETE SUMMARY")
print("="*60)

gold_tables = {
    "gold_fact_transactions"  : f"{GOLD_TABLES}/gold_fact_transactions",
    "gold_merchant_performance": f"{GOLD_TABLES}/gold_merchant_performance",
    "gold_fraud_summary"      : f"{GOLD_TABLES}/gold_fraud_summary",
    "gold_settlement_summary" : f"{GOLD_TABLES}/gold_settlement_summary"
}

total = 0
for table_name, path in gold_tables.items():
    cnt = spark.read.format("delta").load(path).count()
    total += cnt
    print(f"{table_name:<35} {cnt:>10,} rows")

print("="*60)
print(f"{'TOTAL':<35} {total:>10,} rows")
print(f"Completed at: {datetime.now()}")
print("="*60)

StatementMeta(, 876dd5de-bf96-4a12-8cce-079ce232d822, 13, Finished, Available, Finished, False)


GOLD LAYER COMPLETE SUMMARY
gold_fact_transactions                 145,986 rows
gold_merchant_performance                2,000 rows
gold_fraud_summary                       2,680 rows
gold_settlement_summary                  9,207 rows
TOTAL                                  159,873 rows
Completed at: 2026-05-06 01:20:04.750560


In [12]:
# ── Register Gold tables so they appear in Lakehouse ────────
gold_tables = [
    "gold_fact_transactions",
    "gold_merchant_performance",
    "gold_fraud_summary",
    "gold_settlement_summary"
]

for table in gold_tables:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS
        Interac_Fabric_Workspace.Interac_Gold.dbo.{table}
        USING DELTA
        LOCATION '{GOLD_TABLES}/{table}'
    """)
    print(f"Registered: {table} ✅")

print("\nAll Gold tables registered successfully")

StatementMeta(, 876dd5de-bf96-4a12-8cce-079ce232d822, 14, Finished, Available, Finished, False)

Registered: gold_fact_transactions ✅
Registered: gold_merchant_performance ✅
Registered: gold_fraud_summary ✅
Registered: gold_settlement_summary ✅

All Gold tables registered successfully
